<a href="https://colab.research.google.com/github/arushikumarics24-beep/BIS_LAB/blob/main/optimization_via_gene_expression_week8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np

# Generator parameters
Pmin = np.array([50, 50, 50])
Pmax = np.array([200, 250, 300])

a = np.array([0.003, 0.002, 0.0015])
b = np.array([5.3, 5.5, 5.8])
c = np.array([500, 400, 200])

demand = 450

# Algorithm parameters
POPULATION = 100
THETA = 0.25


# Repair solution
def repair(P):
    P = np.clip(P, Pmin, Pmax)

    diff = demand - np.sum(P)

    while abs(diff) > 1e-8:
        if diff > 0:
            available = Pmax - P
        else:
            available = P - Pmin

        indices = np.where(available > 1e-8)[0]

        if len(indices) == 0:
            break

        change = diff / len(indices)

        for i in indices:
            amount = min(abs(change), available[i])
            P[i] += amount if diff > 0 else -amount

        diff = demand - np.sum(P)

    return P


# Calculate generation cost
def fitness(P):
    return np.sum(a * P**2 + b * P + c)


# Step 1: Create primary graph of genes
genes = np.array([
    repair(np.random.uniform(Pmin, Pmax))
    for _ in range(POPULATION)
])

# Step 2: Apply neighborhood threshold
# Normalize the distance using generator capacities
scale = Pmax - Pmin

graph = {
    i: set() for i in range(POPULATION)
}

for i in range(POPULATION):
    for j in range(i + 1, POPULATION):

        distance = np.linalg.norm(
            (genes[i] - genes[j]) / scale
        )

        if distance <= THETA:
            graph[i].add(j)
            graph[j].add(i)


# Step 3: Initialize final gene set
final_genes = []


# Find maximum community using connected components
def find_communities(graph):
    visited = set()
    communities = []

    for node in graph:
        if node not in visited:
            stack = [node]
            community = []
            visited.add(node)

            while stack:
                current = stack.pop()
                community.append(current)

                for neighbor in graph[current]:
                    if neighbor not in visited:
                        visited.add(neighbor)
                        stack.append(neighbor)

            communities.append(community)

    return communities


# Steps 4-8: Community selection and gene extraction
while graph:

    communities = find_communities(graph)

    # Find maximum community
    MC = max(communities, key=len)

    # Select gene with highest importance
    # Lower generation cost = higher importance
    candidate = min(MC, key=lambda i: fitness(genes[i]))

    # Add candidate gene to final gene set
    final_genes.append(candidate)

    # Remove processed community from graph
    for node in MC:
        del graph[node]

    for node in graph:
        graph[node] -= set(MC)


# Step 9: Report final gene set
best = min(
    (genes[i] for i in final_genes),
    key=fitness
)

print("Gene Expression Algorithm")
print("Generator Outputs:", np.round(best, 2))
print("Total Power:", round(np.sum(best), 2), "MW")
print("Minimum Generation Cost:", round(fitness(best), 2))
print("Selected Genes:", len(final_genes))


Gene Expression Algorithm
Generator Outputs: [155.43 169.25 125.32]
Total Power: 450.0 MW
Minimum Generation Cost: 3734.83
Selected Genes: 1
